In [9]:
import json
import pandas as pd

# 1. Wczytanie pliku JSON źródłowego
file_path = '..\\data\\raw\\match_1821123_Southampton_vs_Aston_Villa_raw.json'
with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

match_id = "1821123"   # lub wyciągnięte dynamicznie z nazwy pliku

# ==========================================
# TABELA 1: Match Info & Teams (Mecze i Drużyny)
# ==========================================
teams_data = []
for side in ['home', 'away']:
    team = data.get(side)
    if team:
        row = {
            'match_id': match_id,
            'side': side,
            'team_id': team.get('teamId'),
            'name': team.get('name'),
            'manager_name': team.get('managerName'),
            'country_name': team.get('countryName'),
            'average_age': team.get('averageAge'),
            'attendance': data.get('attendance'),
            'elapsed': data.get('elapsed'),
            'et_score': data.get('etScore'),
            'score_fulltime': team.get('scores', {}).get('fulltime'),
            'score_halftime': team.get('scores', {}).get('halftime'),
            'score_running': team.get('scores', {}).get('running'),
        }
        teams_data.append(row)
df_teams = pd.DataFrame(teams_data)


# ==========================================
# TABELA 2: Players (Zawodnicy - dane bazowe)
# ==========================================
players_data = []
player_stats_minute_data = []  # Tutaj trafią szczegółowe statystyki minutowe, żeby nic nie zgubić

for side in ['home', 'away']:
    team = data.get(side)
    if team and 'players' in team:
        for p in team['players']:
            # Podstawowe dane gracza
            p_row = {
                'match_id': match_id,
                'team_id': team.get('teamId'),
                'player_id': p.get('playerId'),
                'name': p.get('name'),
                'position': p.get('position'),
                'shirt_no': p.get('shirtNo'),
                'age': p.get('age'),
                'height': p.get('height'),
                'weight': p.get('weight'),
                'is_first_eleven': p.get('isFirstEleven'),
                'is_man_of_the_match': p.get('isManOfTheMatch'),
                'subbed_in_minute': p.get('subbedInExpandedMinute'),
                'subbed_out_minute': p.get('subbedOutExpandedMinute')
            }
            players_data.append(p_row)
            
            # WYCIĄGANIE WSZYSTKICH STATYSTYK MINUTOWYCH (np. ratings, touches, passesAccurate itp.)
            stats = p.get('stats', {})
            for stat_name, minute_dict in stats.items():
                if isinstance(minute_dict, dict):
                    for minute_str, val in minute_dict.items():
                        player_stats_minute_data.append({
                            'match_id': match_id,
                            'player_id': p.get('playerId'),
                            'stat_metric': stat_name,
                            'minute': int(minute_str),
                            'value': val
                        })

df_players = pd.DataFrame(players_data)
df_player_stats_minutes = pd.DataFrame(player_stats_minute_data)


# ==========================================
# TABELA 3 & 4: Events & Event Qualifiers (Zdarzenia i ich kwalifikatory)
# ==========================================
# Zbieramy główne 'events' oraz 'incidentEvents' z sekcji drużyny
all_raw_events = data.get('events', [])
if 'away' in data and 'incidentEvents' in data['away']:
    all_raw_events.extend(data['away']['incidentEvents'])
if 'home' in data and 'incidentEvents' in data['home']:
    all_raw_events.extend(data['home']['incidentEvents'])

events_rows = []
qualifiers_rows = []

for ev in all_raw_events:
    ev_id = ev.get('id')
    
    # Główny wiersz zdarzenia[cite: 1]
    events_rows.append({
        'match_id': match_id,
        'event_id': ev_id,
        'event_index_id': ev.get('eventId'),
        'team_id': ev.get('teamId'),
        'player_id': ev.get('playerId'),
        'related_player_id': ev.get('relatedPlayerId'),
        'minute': ev.get('minute'),
        'second': ev.get('second'),
        'expanded_minute': ev.get('expandedMinute'),
        'period_value': ev.get('period', {}).get('value'),
        'period_name': ev.get('period', {}).get('displayName'),
        'type_value': ev.get('type', {}).get('value'),
        'type_name': ev.get('type', {}).get('displayName'),
        'outcome_value': ev.get('outcomeType', {}).get('value'),
        'outcome_name': ev.get('outcomeType', {}).get('displayName'),
        'x': ev.get('x'),
        'y': ev.get('y'),
        'end_x': ev.get('endX'),
        'end_y': ev.get('endY'),
        'goal_mouth_y': ev.get('goalMouthY'),
        'goal_mouth_z': ev.get('goalMouthZ'),
        'is_touch': ev.get('isTouch'),
        'is_shot': ev.get('isShot'),
        'is_goal': ev.get('isGoal'),
        'text': ev.get('text')
    })
    
    # Wyciąganie podrzędnych 'qualifiers' dla każdego zdarzenia (zabezpieczenie przed utratą detali)[cite: 1]
    qualifiers = ev.get('qualifiers', [])
    for q in qualifiers:
        q_type = q.get('type', {})
        qualifiers_rows.append({
            'match_id': match_id,
            'event_id': ev_id,
            'qualifier_type_value': q_type.get('value'),
            'qualifier_type_name': q_type.get('displayName'),
            'qualifier_value': q.get('value')
        })

df_events = pd.DataFrame(events_rows)
df_event_qualifiers = pd.DataFrame(qualifiers_rows)


# ==========================================
# TABELA 5 & 6: Formations & Formation Positions (Formacje taktyczne)
# ==========================================
formations_rows = []
formation_positions_rows = []

for side in ['home', 'away']:
    team = data.get(side)
    if team and 'formations' in team:
        for f in team['formations']:
            f_id = f.get('formationId')
            start_min = f.get('startMinuteExpanded')
            
            formations_rows.append({
                'match_id': match_id,
                'team_id': team.get('teamId'),
                'formation_id': f_id,
                'formation_name': f.get('formationName'),
                'captain_player_id': f.get('captainPlayerId'),
                'start_minute': start_min,
                'end_minute': f.get('endMinuteExpanded'),
                'period': f.get('period')
            })
            
            # Pozycje współrzędnych formacji[cite: 1]
            positions = f.get('formationPositions', [])
            for idx, pos in enumerate(positions):
                formation_positions_rows.append({
                    'match_id': match_id,
                    'team_id': team.get('teamId'),
                    'formation_start_minute': start_min,
                    'slot_index': idx,
                    'horizontal': pos.get('horizontal'),
                    'vertical': pos.get('vertical')
                })

df_formations = pd.DataFrame(formations_rows)
df_formation_positions = pd.DataFrame(formation_positions_rows)

In [10]:
# Przykład łatwego wyboru kolumn do nowej analizy
df_subset = database['events'][
    ['match_id', 'minute', 'expandedMinute', 'type.displayName', 'x', 'y']
]
df_subset.head()

,match_id,minute,expandedMinute,type.displayName,x,y
0,1821123,0,0,Start,0.0,0.0
1,1821123,0,0,Start,0.0,0.0
2,1821123,0,0,Pass,50.0,50.0
3,1821123,0,0,Pass,45.6,49.9
4,1821123,0,0,Pass,43.0,43.9
